In [5]:
import pandas as pd

# Load from local CSV (you uploaded this file)
per_game = pd.read_csv("nba_2023-24.csv")

# Remove repeated header rows
per_game = per_game[per_game['Rk'] != 'Rk']
per_game.reset_index(drop=True, inplace=True)

# Clean column names
per_game.columns = per_game.columns.str.strip()

# Rename 'Team' to 'Tm' if needed
if 'Team' in per_game.columns:
    per_game.rename(columns={'Team': 'Tm'}, inplace=True)

# Define the core stats we're interested in
expected_cols = ['Player', 'Age', 'Tm', 'Pos', 'G', 'GS', 'MP', 'PTS', 'AST', 'TRB', 'STL', 'BLK']
available_cols = [col for col in expected_cols if col in per_game.columns]
per_game = per_game[available_cols]

# Normalize player names (remove accents, whitespace)
per_game['Player'] = per_game['Player'].str.normalize('NFKD') \
                                       .str.encode('ascii', errors='ignore') \
                                       .str.decode('utf-8') \
                                       .str.strip()

# ✅ Preview cleaned per-game data
per_game.head()


,Player,Age,Tm,Pos,G,GS,MP,PTS,AST,TRB,STL,BLK
0,Joel Embiid,29.0,PHI,C,39.0,39.0,33.6,34.7,5.6,11.0,1.2,1.7
1,Luka Doncic,24.0,DAL,PG,70.0,70.0,37.5,33.9,9.8,9.2,1.4,0.5
2,Giannis Antetokounmpo,29.0,MIL,PF,73.0,73.0,35.2,30.4,6.5,11.5,1.2,1.1
3,Shai Gilgeous-Alexander,25.0,OKC,PG,75.0,75.0,34.0,30.1,6.2,5.5,2.0,0.9
4,Jalen Brunson,27.0,NYK,PG,77.0,77.0,35.4,28.7,6.7,3.6,0.9,0.2


In [18]:
import pandas as pd

# ✅ Load advanced stats from uploaded CSV
advanced = pd.read_csv("2023-24_advanced.csv")

# ✅ Drop repeated header rows from Basketball-Reference exports
advanced = advanced[advanced['Rk'] != 'Rk']

# ✅ Convert important numeric fields
advanced[['WS', 'VORP']] = advanced[['WS', 'VORP']].apply(pd.to_numeric, errors='coerce')

# ✅ Merge with per_game from Cell 1
merged_stats = pd.merge(per_game, advanced, on='Player', how='left')

# ✅ Rename columns for clarity
merged_stats.rename(columns={
    'Age_x': 'Age',
    'Pos_x': 'Pos',
    'MP_x': 'MP',
}, inplace=True)

# ✅ Keep only useful columns
selected_cols = ['Player', 'Age', 'Tm', 'Pos', 'G_x', 'MP', 'PTS', 'AST', 'TRB', 'WS', 'VORP']
merged_stats = merged_stats[selected_cols]

# ✅ Preview final output
merged_stats.head()




,Player,Age,Tm,Pos,G_x,MP,PTS,AST,TRB,WS,VORP
0,Joel Embiid,29.0,PHI,C,39.0,33.6,34.7,5.6,11.0,7.5,4.5
1,Luka Doncic,24.0,DAL,PG,70.0,37.5,33.9,9.8,9.2,NaN,NaN
2,Giannis Antetokounmpo,29.0,MIL,PF,73.0,35.2,30.4,6.5,11.5,13.2,7.2
3,Shai Gilgeous-Alexander,25.0,OKC,PG,75.0,34.0,30.1,6.2,5.5,14.6,7.1
4,Jalen Brunson,27.0,NYK,PG,77.0,35.4,28.7,6.7,3.6,11.2,5.4


In [ ]:
#Add salary and contract data

# Example dictionary of salaries and contract years left
salary_data = {
    'Joel Embiid': (47600000, 3),
    'Luka Doncic': (40800000, 4),
    'Giannis Antetokounmpo': (45600000, 3),
    'Shai Gilgeous-Alexander': (33000000, 4),
    'Jalen Brunson': (26000000, 2),
    # Add more players here...
}

# Create DataFrame from dictionary
salary_df = pd.DataFrame.from_dict(salary_data, orient='index', columns=['Salary', 'ContractYearsLeft'])
salary_df.reset_index(inplace=True)
salary_df.rename(columns={'index': 'Player'}, inplace=True)

# Merge salary data into merged_stats
merged_stats = pd.merge(merged_stats, salary_df, on='Player', how='left')

# Confirm addition
merged_stats[['Player', 'Salary', 'ContractYearsLeft']].head()



,Player,Salary,ContractYearsLeft
0,Joel Embiid,47600000.0,3.0
1,Luka Doncic,40800000.0,4.0
2,Giannis Antetokounmpo,45600000.0,3.0
3,Shai Gilgeous-Alexander,33000000.0,4.0
4,Jalen Brunson,26000000.0,2.0


In [28]:
# Cell 4: Add a basic undervalue score (e.g., VORP per $Million)

filtered = merged_stats[(merged_stats['Salary'].notna()) & (merged_stats['VORP'].notna())]
filtered['ValueScore'] = filtered['VORP'] / (filtered['Salary'] / 1_000_000)
undervalued_players = filtered.sort_values(by='ValueScore', ascending=False)

undervalued_players[['Player', 'VORP', 'Salary', 'ContractYearsLeft', 'ValueScore']].head(10)

C:\Users\mikey\AppData\Local\Temp\ipykernel_23424\3644794932.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered['ValueScore'] = filtered['VORP'] / (filtered['Salary'] / 1_000_000)


,Player,VORP,Salary,ContractYearsLeft,ValueScore
3,Shai Gilgeous-Alexander,7.1,33000000.0,4.0,0.215152
4,Jalen Brunson,5.4,26000000.0,2.0,0.207692
2,Giannis Antetokounmpo,7.2,45600000.0,3.0,0.157895
0,Joel Embiid,4.5,47600000.0,3.0,0.094538


In [4]:
import pandas as pd

teams = ['ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW',
         'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK',
         'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']

all_heights = []

for team in teams:
    url = f"https://www.basketball-reference.com/teams/{team}/2024.html"
    try:
        tables = pd.read_html(url)
        roster = tables[0]  # Roster table is always first
        if 'Player' in roster.columns and 'Ht' in roster.columns:
            roster = roster[['Player', 'Ht']]
            roster['Team'] = team
            all_heights.append(roster)
    except Exception as e:
        print(f"⚠️ Failed to load {team}: {e}")

# Combine all team data into one DataFrame
height_df = pd.concat(all_heights, ignore_index=True)

# Normalize player names
height_df['Player'] = height_df['Player'].str.normalize('NFKD') \
                                         .str.encode('ascii', errors='ignore') \
                                         .str.decode('utf-8') \
                                         .str.strip()

# Preview result
print(f"✅ Loaded heights for {len(height_df)} players")
height_df.head()


C:\Users\mikey\AppData\Local\Temp\ipykernel_16108\1711175004.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  roster['Team'] = team
C:\Users\mikey\AppData\Local\Temp\ipykernel_16108\1711175004.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  roster['Team'] = team
C:\Users\mikey\AppData\Local\Temp\ipykernel_16108\1711175004.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the 

⚠️ Failed to load UTA: HTTP Error 429: Too Many Requests
⚠️ Failed to load WAS: HTTP Error 429: Too Many Requests
✅ Loaded heights for 612 players


,Player,Ht,Team
0,Saddiq Bey,6-7,ATL
1,Bogdan Bogdanovic,6-5,ATL
2,Kobe Bufkin,6-4,ATL
3,Clint Capela,6-10,ATL
4,Bruno Fernando,6-9,ATL
